In [17]:
import os
import time
from dotenv import load_dotenv
from notion_client import Client
from notion_client.errors import APIResponseError

load_dotenv()

notion = Client(auth=os.getenv("NOTION_API_KEY"))

print("Notion client initialized")

# Notion allows ~3 requests/sec sustained. A small fixed delay between
# calls avoids tripping the limit in the first place.
REQUEST_DELAY = 0.35  # seconds
MAX_RETRIES = 5


def call_with_retry(fn, **kwargs):
    """Call a Notion API function, retrying on 429 rate-limit errors."""
    for attempt in range(MAX_RETRIES):
        try:
            result = fn(**kwargs)
            time.sleep(REQUEST_DELAY)  # gentle pacing between calls
            return result
        except APIResponseError as e:
            if e.status == 429:
                # Notion tells us how long to wait via Retry-After (seconds)
                retry_after = float(e.headers.get("Retry-After", 1))
                print(f"    Rate limited. Waiting {retry_after}s (attempt {attempt + 1}/{MAX_RETRIES})...")
                time.sleep(retry_after)
                continue
            raise  # some other API error, don't swallow it
    raise RuntimeError("Max retries exceeded due to repeated rate limiting")


def get_all_pages():
    """Paginate through notion.search() to get every page/database."""
    pages = []
    cursor = None
    while True:
        kwargs = {"query": ""}
        if cursor:
            kwargs["start_cursor"] = cursor
        response = call_with_retry(notion.search, **kwargs)
        pages.extend(response["results"])
        if response.get("has_more"):
            cursor = response.get("next_cursor")
        else:
            break
    return pages


def extract_text(block):
    block_type = block["type"]
    if block_type not in block:
        return ""
    content = block[block_type]
    rich_text = content.get("rich_text", [])
    return "".join(item.get("plain_text", "") for item in rich_text)


def get_all_blocks(block_id):
    """Paginate through children, recursing into nested blocks."""
    all_blocks = []
    cursor = None
    while True:
        kwargs = {"block_id": block_id}
        if cursor:
            kwargs["start_cursor"] = cursor
        resp = call_with_retry(notion.blocks.children.list, **kwargs)
        for block in resp["results"]:
            all_blocks.append(block)
            if block.get("has_children"):
                # recurse into nested content (toggles, sub-lists, etc.)
                all_blocks.extend(get_all_blocks(block["id"]))
        if resp.get("has_more"):
            cursor = resp.get("next_cursor")
        else:
            break
    return all_blocks


def get_page_title(page):
    """Best-effort extraction of a page's title from its properties."""
    props = page.get("properties", {})
    for prop in props.values():
        if prop.get("type") == "title":
            return "".join(t.get("plain_text", "") for t in prop.get("title", []))
    return page.get("id", "untitled")


pages = get_all_pages()
print(f"Found {len(pages)} pages/databases")

all_text_parts = []

for i, page in enumerate(pages):
    if page["object"] != "page":
        # skip databases themselves; their child pages are returned separately
        continue

    title = get_page_title(page)
    print(f"[{i}] Fetching: {title} ({page['id']})")

    blocks = get_all_blocks(page["id"])
    print(f"    {len(blocks)} blocks found")

    page_lines = [f"=== PAGE: {title} ===", f"URL: {page.get('url', '')}", ""]

    for block in blocks:
        text = extract_text(block)
        if text:
            page_lines.append(text)

    all_text_parts.append("\n".join(page_lines))

full_document = "\n\n".join(all_text_parts)

with open("notion_data.txt", "w", encoding="utf-8") as f:
    f.write(full_document)

print(f"Saved {len(pages)} pages to notion_data.txt")

Notion client initialized
Found 46 pages/databases
[0] Fetching: Agent Evaluation (3d5fca33-84fc-80fe-aac2-d743eaa88e60)
    55 blocks found
[1] Fetching: AI Roadmap (2d9fca33-84fc-805f-93da-fe1f3c5b7277)
    8544 blocks found
[2] Fetching: Training LLM from Scratch (3d2fca33-84fc-808d-a62d-ffc553144ab8)
    369 blocks found
[3] Fetching: AutoGen (3b6fca33-84fc-80db-a710-da50ab997c7f)
    115 blocks found
[4] Fetching: Fine-Tuning LLM (3b2fca33-84fc-80d6-8dcc-ec22d752d14d)
    583 blocks found
[5] Fetching: Multi-Agent Architecture (3b0fca33-84fc-8035-b72a-fcd1b4ae9925)
    4 blocks found
[6] Fetching: quick questions (3abfca33-84fc-8038-81e6-e924cc3dedf6)
    262 blocks found
[7] Fetching: React JS (289fca33-84fc-801c-acfd-eadfa7126d4c)
    416 blocks found
[8] Fetching: Rag Theory (3a9fca33-84fc-80b5-832e-eb5d0382e463)
    328 blocks found
[9] Fetching: python (274fca33-84fc-8031-bbb6-d2d0870d18e0)
    342 blocks found
[10] Fetching: Python Interview Questions (335fca33-84fc-80b0-859